# DemonTTS L4 Training — 24GB VRAM Optimization

Run this on **Google Colab with L4 GPU** (Runtime → Change runtime type → GPU → L4).

**VRAM Budget:** 24GB
- Faraday: ~2GB fp16 + ~4GB activations @ B=4 → ~8GB
- Aether: ~180MB fp16 + ~2GB activations @ B=16 → ~4GB
- Total training: <16GB, leaves headroom for data loading


In [ ]:
# === CELL 1: Verify L4 GPU ===
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

assert 'L4' in torch.cuda.get_device_name(0), "ERROR: Not running on L4! Change runtime type."
print("\n✓ L4 confirmed. 24GB VRAM available.")

In [ ]:
# === CELL 2: Mount Google Drive for persistent checkpoints ===
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory in Drive
!mkdir -p /content/drive/MyDrive/demon-tts/checkpoints
!mkdir -p /content/drive/MyDrive/demon-tts/data
print("Drive mounted. Checkpoints will persist in /content/drive/MyDrive/demon-tts/")

In [ ]:
# === CELL 3: Clone repo & install deps ===
!git clone https://github.com/seal/demon-tts.git /content/demon-tts 2>/dev/null || echo "Repo exists"
%cd /content/demon-tts

# Install optimized deps for L4
!pip install -q transformers accelerate bitsandbytes torchaudio soundfile numpy scipy matplotlib

# Verify installs
import torch
print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## Data Upload

**Option A (Fastest):** Upload your `data/aether_pairs/` and `data/faraday_pairs/` to Google Drive at `MyDrive/demon-tts/data/`, then run Cell 4A.

**Option B:** Generate fresh data on Colab (slow, ~2 hours for 5000 pairs). Run Cell 4B instead.

**Option C:** Use a small subset for quick validation. Run Cell 4C.

In [ ]:
# === CELL 4A: Copy data from Drive (RECOMMENDED) ===
!mkdir -p data/aether_pairs data/faraday_pairs
!cp -r /content/drive/MyDrive/demon-tts/data/aether_pairs/* data/aether_pairs/ 2>/dev/null
!cp -r /content/drive/MyDrive/demon-tts/data/faraday_pairs/* data/faraday_pairs/ 2>/dev/null

import os
aether_count = len([f for f in os.listdir('data/aether_pairs') if f.endswith('.pt')])
faraday_count = len([f for f in os.listdir('data/faraday_pairs') if f.endswith('.pt')])
print(f"Aether pairs: {aether_count}")
print(f"Faraday pairs: {faraday_count}")

if aether_count == 0 or faraday_count == 0:
    print("\n⚠️ No data found in Drive! Use Option B or C below.")
else:
    print("\n✓ Data loaded from Drive.")

In [ ]:
# === CELL 4B: Generate data on Colab (SLOW - skip if using 4A) ===
# !python generate_training_data.py --n_pairs 2000 --output_dir data

In [ ]:
# === CELL 4C: Use tiny subset for smoke test (skip if using 4A) ===
# !mkdir -p data/aether_pairs data/faraday_pairs
# !python generate_training_data.py --n_pairs 100 --output_dir data

## Phase 1: Faraday Training (Mel Enhancement)

**L4 Optimizations:**
- Batch size 4 (vs 1 on RTX 4060)
- No gradient checkpointing (saves 20% overhead)
- Mixed precision (fp16)
- `torch.compile()` for 1.5-2x speedup
- 100 epochs target (~3-4 hours on L4)

In [ ]:
# === CELL 5: Faraday Training Config ===
FARADAY_CONFIG = {
    'text_dim': 512,
    'speaker_dim': 512,
    'cond_dim': 512,
    'base_channels': 192,  # 509M params
    'batch_size': 4,
    'grad_accum': 2,  # effective B=8
    'lr': 2e-4,
    'epochs': 100,
    'checkpoint_dir': '/content/drive/MyDrive/demon-tts/checkpoints/faraday',
    'use_compile': True,
    'use_checkpointing': False,  # L4 has enough VRAM
}

!mkdir -p {FARADAY_CONFIG['checkpoint_dir']}
print("Faraday config ready.")
print(f"Effective batch size: {FARADAY_CONFIG['batch_size'] * FARADAY_CONFIG['grad_accum']}")
print(f"Checkpoint dir: {FARADAY_CONFIG['checkpoint_dir']}")

In [ ]:
# === CELL 6: Train Faraday ===
import torch
import sys
sys.path.insert(0, '/content/demon-tts')

from training.train_faraday_supervised import main as train_faraday

# Override args for L4
import argparse
args = argparse.Namespace(
    data_dir='data/faraday_pairs',
    output_dir=FARADAY_CONFIG['checkpoint_dir'],
    epochs=FARADAY_CONFIG['epochs'],
    batch_size=FARADAY_CONFIG['batch_size'],
    lr=FARADAY_CONFIG['lr'],
    grad_accum=FARADAY_CONFIG['grad_accum'],
    resume=None,
)

# Monkey-patch for L4 optimizations
import training.train_faraday_supervised as ft
original_main = ft.main

def l4_main():
    device = torch.device('cuda')
    
    from faraday.model import FaradayDiffusion
    model = FaradayDiffusion(
        text_dim=512,
        speaker_dim=512,
        cond_dim=512,
        base_channels=192,
    ).to(device)
    
    # L4: disable checkpointing for speed
    model.unet.use_checkpoint = False
    for m in model.unet.modules():
        if hasattr(m, 'checkpoint'):
            m.checkpoint = False
    
    # torch.compile for 1.5-2x speed
    if FARADAY_CONFIG['use_compile']:
        print("[L4] Compiling model with torch.compile...")
        model = torch.compile(model, mode='reduce-overhead')
    
    print(f"[L4] Model loaded: {sum(p.numel() for p in model.parameters()):,} params")
    print(f"[L4] VRAM after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    
    # Continue with training...
    return original_main()

ft.main = l4_main
train_faraday()

## Phase 2: Aether Training (Waveform Refinement)

**L4 Optimizations:**
- Batch size 16 (vs 1 on RTX 4060)
- Transformer architecture (91M params)
- No gradient checkpointing needed
- `torch.compile()` enabled
- 50 epochs target (~2 hours on L4)

In [ ]:
# === CELL 7: Aether Training Config ===
AETHER_CONFIG = {
    'batch_size': 16,
    'grad_accum': 1,
    'lr': 1e-4,
    'epochs': 50,
    'checkpoint_dir': '/content/drive/MyDrive/demon-tts/checkpoints/aether',
    'use_compile': True,
}

!mkdir -p {AETHER_CONFIG['checkpoint_dir']}
print("Aether config ready.")
print(f"Batch size: {AETHER_CONFIG['batch_size']}")
print(f"Checkpoint dir: {AETHER_CONFIG['checkpoint_dir']}")

In [ ]:
# === CELL 8: Train Aether ===
import sys
sys.path.insert(0, '/content/demon-tts')

from training.train_aether_supervised import main as train_aether
import argparse

args = argparse.Namespace(
    data_dir='data/aether_pairs',
    output_dir=AETHER_CONFIG['checkpoint_dir'],
    epochs=AETHER_CONFIG['epochs'],
    batch_size=AETHER_CONFIG['batch_size'],
    lr=AETHER_CONFIG['lr'],
    grad_accum=AETHER_CONFIG['grad_accum'],
    resume=None,
)

# Monkey-patch for L4
import training.train_aether_supervised as at
original_aether_main = at.main

def l4_aether_main():
    device = torch.device('cuda')
    
    from aether.model import AetherFilter
    model = AetherFilter(lr=AETHER_CONFIG['lr']).to(device)
    
    if AETHER_CONFIG['use_compile']:
        print("[L4] Compiling Aether with torch.compile...")
        model = torch.compile(model, mode='reduce-overhead')
    
    print(f"[L4] Aether params: {sum(p.numel() for p in model.filter_net.parameters()):,}")
    print(f"[L4] VRAM after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    
    return original_aether_main()

at.main = l4_aether_main
train_aether()

## Phase 3: Inference Test

Generate audio with trained checkpoints to verify quality.

In [ ]:
# === CELL 9: Copy best checkpoints to models/ ===
import shutil
import os

# Copy from Drive checkpoints to local models/
faraday_ckpt = f"{FARADAY_CONFIG['checkpoint_dir']}/best.pt"
aether_ckpt = f"{AETHER_CONFIG['checkpoint_dir']}/best.pt"

if os.path.exists(faraday_ckpt):
    shutil.copy(faraday_ckpt, 'models/faraday.pt')
    print(f"✓ Faraday: {faraday_ckpt}")
else:
    print("⚠️ Faraday best.pt not found yet. Train first.")

if os.path.exists(aether_ckpt):
    shutil.copy(aether_ckpt, 'models/aether.pt')
    print(f"✓ Aether: {aether_ckpt}")
else:
    print("⚠️ Aether best.pt not found yet. Train first.")

In [ ]:
# === CELL 10: Generate test audio ===
from demo_tts import DemonTTS
from IPython.display import Audio, display

tts = DemonTTS()

text = "The demon speaks through the void, his voice carved from static and starlight."

# Full pipeline
wav = tts.synthesize(text, voice_id='Rick C-137', use_faraday=True, use_aether=True)
tts.save_wav(wav, '/content/drive/MyDrive/demon-tts/output_l4.wav')

print(f"Generated {len(wav)/24000:.2f}s audio")
display(Audio(wav, rate=24000))

## Advanced: Resume Training

If Colab disconnects, resume from last checkpoint.

In [ ]:
# === CELL 11: Resume Faraday from checkpoint ===
# Find latest emergency checkpoint
import glob
ckpts = glob.glob(f"{FARADAY_CONFIG['checkpoint_dir']}/*emergency.pt")
if ckpts:
    latest = sorted(ckpts)[-1]
    print(f"Resuming Faraday from: {latest}")
    # Pass --resume to training script
else:
    print("No checkpoint found to resume.")